## Primer vistazo

In [2]:
from pathlib import Path
import duckdb

con = duckdb.connect()  # base en memoria, por ahora

con.sql("SET TimeZone = 'UTC';")

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

RAW_PATH = (BASE_DIR / "data" / "raw").as_posix()
BRONZE_PATH = (BASE_DIR / "data" / "bronze").as_posix()
SILVER_PATH = (BASE_DIR / "data" / "silver").as_posix()


In [ ]:
con.sql(f"SELECT * FROM read_csv('{RAW_PATH}/clientes.csv', all_varchar=true) LIMIT 5").show()

In [ ]:
# SUMMARIZE PARA UN APROXIMADO
# Lo que SUMMARIZE no te muestra
# Ids duplicados de forma exacta.
# Integridad entre tablas: por ejemplo, pedidos con un cliente_id que no existe en clientes.
# Filas raras: como los registros casi vacíos del final, que solo ves mirando los datos.
# Valores vacíos disfrazados: un texto vacío '' o 'N/A' no cuenta como nulo, pero es un problema igual.

con.sql(f"""
    SELECT column_name, column_type, count, null_percentage, approx_unique
    FROM (SUMMARIZE SELECT * FROM read_csv('{RAW_PATH}/clientes.csv', all_varchar=true))
""").show()

In [ ]:
# Revisar si existen id unicos

con.sql(f"""
    SELECT COUNT(*) AS filas,
           COUNT(DISTINCT id) AS ids_distintos
    FROM read_csv('{RAW_PATH}/clientes.csv', all_varchar=true)
""").show()

In [ ]:
# Nulos aproximados

con.sql(f"""
    SELECT COUNT(*) - COUNT(email) AS nulos_email
    FROM read_csv('{RAW_PATH}/clientes.csv', all_varchar=true)
""").show()

--------

In [3]:
con.sql(f"SELECT * FROM '{BRONZE_PATH}/pedidos.parquet' LIMIT 5").show()


┌─────────┬────────────┬─────────────┬──────────┬────────────┬─────────┬────────────┬───────────────────────────────┬──────────────┐
│   id    │ cliente_id │ producto_id │ cantidad │   fecha    │ importe │   estado   │         _ingested_at          │ _source_file │
│ varchar │  varchar   │   varchar   │ varchar  │  varchar   │ varchar │  varchar   │   timestamp with time zone    │   varchar    │
├─────────┼────────────┼─────────────┼──────────┼────────────┼─────────┼────────────┼───────────────────────────────┼──────────────┤
│ 1       │ 147        │ 18          │ 4        │ 2023-09-17 │ 179.19  │ completado │ 2026-11-23 14:43:37.001853+00 │ pedidos.csv  │
│ 2       │ 2881       │ 35          │ 2        │ 2024-05-19 │ 93.28   │ completado │ 2026-11-23 14:43:37.001853+00 │ pedidos.csv  │
│ 3       │ 821        │ 52          │ 5        │ 2023-11-29 │ 44.01   │ devuelto   │ 2026-11-23 14:43:37.001853+00 │ pedidos.csv  │
│ 4       │ 258        │ 69          │ 3        │ 2023-11-05 │ 18.18 

------

In [ ]:
con.sql(
f"""
SELECT 
    TRY_CAST(id AS INTEGER) AS id,
    nombre,
    email,
    ciudad,
    TRY_CAST(fecha_registro AS DATE) AS fecha_registro,
    canal_adquisicion, 
    TRY_CAST(es_vip AS BOOLEAN) AS es_vip,
    _ingested_at,
    _source_file
FROM '{BRONZE_PATH}/clientes.parquet' LIMIT 5
"""
).show()

In [10]:
con.sql(
f"""
SELECT es_vip, COUNT(*) AS filas
FROM '{BRONZE_PATH}/clientes.parquet'
GROUP BY es_vip
ORDER BY filas DESC
"""
).show()

┌─────────┬───────┐
│ es_vip  │ filas │
│ varchar │ int64 │
├─────────┼───────┤
│ false   │  4260 │
│ true    │   582 │
│ NULL    │   162 │
└─────────┴───────┘



## Controles de calidad sobre Bronze

Estas consultas ayudan a decidir las transformaciones de Silver. Cuentan nulos y textos vacíos, y muestran distribuciones y rangos sin excluir filas.

In [ ]:
TABLAS_CONTROL = ("clientes", "productos", "pedidos")

def revisar_nulos(tabla):
    if tabla not in TABLAS_CONTROL:
        raise ValueError(f"Tabla fuera del alcance de estos controles: {tabla}")

    ruta = f"{BRONZE_PATH}/{tabla}.parquet".replace("'", "''")
    columnas = [
        fila[0]
        for fila in con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{ruta}')"
        ).fetchall()
        if not fila[0].startswith("_")
    ]

    consultas = []
    for columna in columnas:
        nombre = '"' + columna.replace('"', '""') + '"'
        consultas.append(
            f"""SELECT '{columna}' AS columna,
                       COUNT(*) FILTER (WHERE {nombre} IS NULL) AS nulos,
                       COUNT(*) FILTER (
                           WHERE {nombre} IS NOT NULL
                             AND TRIM(CAST({nombre} AS VARCHAR)) = ''
                       ) AS vacios
                FROM read_parquet('{ruta}')"""
        )

    con.sql(" UNION ALL ".join(consultas) + " ORDER BY columna").show()

revisar_nulos("clientes")
revisar_nulos("productos")
revisar_nulos("pedidos")

### Clientes

Revisar si id y fecha se pueden convertir y cuántos clientes tienen cada valor de `es_vip`. Los campos opcionales vacíos se observan, pero no se convierten en motivo de rechazo en este análisis.

In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) AS filas,
        COUNT(*) FILTER (WHERE TRY_CAST(NULLIF(TRIM(id), '') AS INTEGER) IS NULL) AS id_no_convertible,
        COUNT(*) FILTER (WHERE TRY_CAST(NULLIF(TRIM(fecha_registro), '') AS DATE) IS NULL) AS fecha_no_convertible
    FROM read_parquet('{BRONZE_PATH}/clientes.parquet')
""").show()

con.sql(f"""
    SELECT es_vip, COUNT(*) AS filas
    FROM read_parquet('{BRONZE_PATH}/clientes.parquet')
    GROUP BY es_vip
    ORDER BY filas DESC
""").show()

### Productos

Verificar valores de categoría, precio y stock. Los rangos permiten detectar valores extremos; no son por sí solos reglas para excluir productos.

In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) AS filas,
        COUNT(DISTINCT TRY_CAST(categoria_id AS INTEGER)) AS categorias_distintas,
        COUNT(*) FILTER (WHERE TRY_CAST(NULLIF(TRIM(categoria_id), '') AS INTEGER) IS NULL) AS categoria_no_convertible,
        COUNT(*) FILTER (WHERE TRY_CAST(NULLIF(TRIM(precio), '') AS DECIMAL(10, 2)) IS NULL) AS precio_no_convertible,
        COUNT(*) FILTER (WHERE TRY_CAST(NULLIF(TRIM(stock), '') AS INTEGER) IS NULL) AS stock_no_convertible,
        MIN(TRY_CAST(precio AS DECIMAL(10, 2))) AS precio_minimo,
        MAX(TRY_CAST(precio AS DECIMAL(10, 2))) AS precio_maximo,
        MIN(TRY_CAST(stock AS INTEGER)) AS stock_minimo,
        MAX(TRY_CAST(stock AS INTEGER)) AS stock_maximo,
        AVG(TRY_CAST(stock AS INTEGER)) AS stock_promedio
    FROM read_parquet('{BRONZE_PATH}/productos.parquet')
""").show()

con.sql(f"""
    SELECT categoria_id, COUNT(*) AS productos
    FROM read_parquet('{BRONZE_PATH}/productos.parquet')
    GROUP BY categoria_id
    ORDER BY TRY_CAST(categoria_id AS INTEGER)
""").show()

### Pedidos

Revisar estados, fechas e importes disponibles. Los conteos de nulos se complementan con una muestra de las filas incompletas para entender el contexto.

In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) AS filas,
        COUNT(*) FILTER (WHERE TRY_CAST(NULLIF(TRIM(cliente_id), '') AS INTEGER) IS NULL) AS cliente_id_no_convertible,
        COUNT(*) FILTER (WHERE TRY_CAST(NULLIF(TRIM(producto_id), '') AS INTEGER) IS NULL) AS producto_id_no_convertible,
        COUNT(*) FILTER (WHERE TRY_CAST(NULLIF(TRIM(cantidad), '') AS INTEGER) IS NULL) AS cantidad_no_convertible,
        COUNT(*) FILTER (WHERE TRY_CAST(NULLIF(TRIM(fecha), '') AS DATE) IS NULL) AS fecha_no_convertible,
        COUNT(*) FILTER (WHERE TRY_CAST(NULLIF(TRIM(importe), '') AS DECIMAL(10, 2)) IS NULL) AS importe_no_convertible,
        MIN(TRY_CAST(fecha AS DATE)) AS fecha_minima,
        MAX(TRY_CAST(fecha AS DATE)) AS fecha_maxima,
        MIN(TRY_CAST(importe AS DECIMAL(10, 2))) AS importe_minimo,
        MAX(TRY_CAST(importe AS DECIMAL(10, 2))) AS importe_maximo
    FROM read_parquet('{BRONZE_PATH}/pedidos.parquet')
""").show()

con.sql(f"""
    SELECT estado, COUNT(*) AS pedidos
    FROM read_parquet('{BRONZE_PATH}/pedidos.parquet')
    GROUP BY estado
    ORDER BY pedidos DESC
""").show()

con.sql(f"""
    SELECT id, cliente_id, producto_id, cantidad, fecha, importe, estado
    FROM read_parquet('{BRONZE_PATH}/pedidos.parquet')
    WHERE cliente_id IS NULL OR producto_id IS NULL OR importe IS NULL OR estado IS NULL
    ORDER BY TRY_CAST(id AS INTEGER) DESC
    LIMIT 10
""").show()